### Notebook for verifying that implemented dynamics work correctly

In [1]:
import importlib
import Model
importlib.reload(Model)
import DeerClass
importlib.reload(DeerClass)
import WolfClass
importlib.reload(WolfClass)
from Model import SpeciesModel

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import solara
from mesa.visualization import SolaraViz, make_space_component
from matplotlib.patches import Circle, Patch
from matplotlib.figure import Figure

Could not check jupyter-widgets extensions.
Traceback (most recent call last):
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 201, in check_jupyter
    python_executable = server_python or get_server_python_executable(silent)
                                         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 155, in get_server_python_executable
    pythons = [getcmdline(server["pid"]) for server in servers]
               ~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 135, in getcmdline
    return subprocess.check_output(["wmic", "process", "get", "commandline", "/format:list"]).split(b"\n")[0].split(b" ")[0].decode("utf-8")
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 4

Could not check jupyter-widgets extensions.
Traceback (most recent call last):
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 201, in check_jupyter
    python_executable = server_python or get_server_python_executable(silent)
                                         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 155, in get_server_python_executable
    pythons = [getcmdline(server["pid"]) for server in servers]
               ~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\solara\checks.py", line 135, in getcmdline
    return subprocess.check_output(["wmic", "process", "get", "commandline", "/format:list"]).split(b"\n")[0].split(b" ")[0].decode("utf-8")
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 4

In [2]:
AGENT_COLOURS = {
    "Deer": "orange",
    "Wolf": "blue",
}
SENSING_RADIUS = {
    "Deer": 1,
    "Wolf": 2,
}

HUNTING_RADIUS = {
    "Wolf": 0.2
}

KILL_RADIUS = {
    "Wolf": 0.02
}
def agent_draw(agent):
    if agent.species in AGENT_COLOURS:
        return {"color": AGENT_COLOURS[agent.species], "size": 5}
    return {"color": "green", "size": 1}

def steps_to_time(step):
    """Function to convert step (mins) to minutes/hours"""
    hours = step // 60 
    mins = step % 60

    return hours, mins



# @solara.component
def SpaceWithSensing(model):
    if isinstance(model, solara.Reactive):
        model = model.value

    fig = Figure(figsize=(8,8), dpi=300)
    ax = fig.subplots()

    species_present = set()

    for agent in model.agents:
        if agent.pos is None:
            continue

        colour = AGENT_COLOURS.get(agent.species, "green")
        sense_colour = "green"
        hunt_colour = "red"

        size = 5 if agent.species in AGENT_COLOURS else 1

        # Draw the agent
        ax.scatter(
            agent.pos[0], agent.pos[1],
            c=colour, s=size * 10, zorder=3
        )

        # Draw sensing radius circle
        if agent.species in SENSING_RADIUS:
            circle = Circle(
                (agent.pos[0], agent.pos[1]),
                radius=SENSING_RADIUS[agent.species],
                edgecolor=colour,
                facecolor=sense_colour,
                alpha=0.1,        # Transparent fill
                linewidth=2,
                linestyle="--",    # Dashed border
                zorder=2
            )
            ax.add_patch(circle)

                # Draw sensing radius circle
        if agent.species in HUNTING_RADIUS:
            hunt_circle = Circle(
                (agent.pos[0], agent.pos[1]),
                radius=HUNTING_RADIUS[agent.species],
                facecolor=hunt_colour,
                alpha=0.7,        # Transparent fill
                zorder=2
            )
            ax.add_patch(hunt_circle)

        species_present.add(agent.species)

    # Build legend with both agent dot and sensing radius
    legend_elements = []
    for species in AGENT_COLOURS:
        if species in species_present:
            legend_elements.append(
                Patch(
                    facecolor=AGENT_COLOURS[species],
                    alpha=0.3,
                    edgecolor=AGENT_COLOURS[species],
                    label=f"{species} (r={SENSING_RADIUS.get(species, '?')})"
                )
            )

    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=14,
        framealpha=0.9
    )

    # Get time
    
    hours, mins = steps_to_time(model.steps)

    # Style
    ax.set_xlim(0, model.space.x_max)
    ax.set_ylim(0, model.space.y_max)
    ax.set_aspect("equal")
    ax.set_title(f"Step: {hours}h {mins}m", fontsize=14)
    ax.set_xlabel("X (km)")
    ax.set_ylabel("Y (km)")
    ax.set_facecolor("#f5f5f0")
    fig.tight_layout()

    with  solara.Column(style={"min-width": "500px", "width": "100%"}):
        solara.FigureMatplotlib(fig, format="png")
    # solara.FigureMatplotlib(fig)


zone_of_repulsion = 0.005  # Move away
zone_of_orientation = 0.75  # Align with heading
zone_of_attraction = 2  # Move towards

# Create a plot to explain the zones of interaction
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
# Add circles for each zone
repulsion_circle = Circle((0, 0), zone_of_repulsion, color='red', alpha=0.5, label='Zone of Repulsion')
orientation_circle = Circle((0, 0), zone_of_orientation, color='blue', alpha=0.5, label='Zone of Orientation')
attraction_circle = Circle((0, 0), zone_of_attraction, color='green', alpha=0.5, label='Zone of Attraction')
ax.add_patch(repulsion_circle)
ax.add_patch(orientation_circle)
ax.add_patch(attraction_circle)
# Add legend
legend_elements = [
    Patch(facecolor='red', edgecolor='red', alpha=0.5, label='Zone of Repulsion'),
    Patch(facecolor='blue', edgecolor='blue', alpha=0.5, label='Zone of Orientation'),
    Patch(facecolor='green', edgecolor='green', alpha=0.5, label='Zone of Attraction')
]
ax.legend(handles=legend_elements)
ax.set_title('Zones of Interaction for Agent Movement', fontsize=16)


In [3]:
# Parameters to view wolf pack dynamics and sensing radius
model_params = {
    "max_steps": 1500,
    "init_predators": 6,
    "init_deer": 8,
    "height": 5,
    "width": 5,
    "step_size": 1/60,  # 1 min per step
    "yearly_sunlight_hours": 365*24,
    "seed": 2,
    "predator": 'Wolf',  # Helper attribute to avoid imports when accessing agent type
    "energy_decrease": 0.002,  # Energy decrease parameter 
    "pack_limit": 12,  # packs will split if too large 
    
    # Options to control complexity of the model
    "use_base": False,
    
    "use_pack_dynamics": True,
    "use_random_movement": False,
    "use_veg": False,
    "given_positions": False, # whether to use random positions or pre-chosen positions (for testing purposes)
    "use_boundary_conditions": True, # whether to use boundary conditions (reflecting off walls) or toroidal space
    # "given_positions": {
    # "Deer": [np.array([5, 5]), np.array([7, 7]), np.array([3, 3]), np.array([6.5, 5.5]), np.array([5, 5]), np.array([7, 7.4]), np.array([3.1, 3.1]), np.array([5.5, 5.5])],
    # "Wolf": [np.array([4, 4]), np.array([6, 6])]
    # },
}

# Create the model
small_world = SpeciesModel(**model_params)

SolaraViz(
    small_world,
    components=[
        # make_space_component(agent_portrayal=agent_draw, backend="matplotlib"),
        SpaceWithSensing
    ],
    model_params=model_params,
    name="Predator-Prey Simulation",
)

C:\Users\dz23282\AppData\Roaming\Python\Python313\site-packages\mesa\mesa_logging.py:112: FutureWarning: The use of the `seed` keyword argument is deprecated, use `rng` instead. No functional changes.
  res = func(*args, **kwargs)


Cannot show ipywidgets in text

In [ ]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, Patch

# Parameters
model_params = {
    "max_steps": 1500,
    "init_predators": 7,
    "init_deer": 30,
    "height": 10,
    "width": 10,
    "step_size": 1/60,
    "yearly_sunlight_hours": 365*24,
    "seed": None,
    "predator": 'Wolf',
    "energy_decrease": 0.002,
    "pack_limit": 12,
    "use_base": False,
    "use_pack_dynamics": True,
    "use_random_movement": False,
    "use_veg": False,
    "given_positions": False,
    "use_boundary_conditions": True,
}
# model_params = {
#     "max_steps": 1500,
#     "init_predators": 6,
#     "init_deer": 8,
#     "height": 5,
#     "width": 5,
#     "step_size": 1/60,  # 1 min per step
#     "yearly_sunlight_hours": 365*24,
#     "seed": 2,
#     "predator": 'Wolf',  # Helper attribute to avoid imports when accessing agent type
#     "energy_decrease": 0.002,  # Energy decrease parameter 
#     "pack_limit": 12,  # packs will split if too large 
    
#     # Options to control complexity of the model
#     "use_base": False,
    
#     "use_pack_dynamics": True,
#     "use_random_movement": False,
#     "use_veg": False,
#     "given_positions": False, # whether to use random positions or pre-chosen positions (for testing purposes)
#     "use_boundary_conditions": True, # whether to use boundary conditions (reflecting off walls) or toroidal space
#     # "given_positions": {
#     # "Deer": [np.array([5, 5]), np.array([7, 7]), np.array([3, 3]), np.array([6.5, 5.5]), np.array([5, 5]), np.array([7, 7.4]), np.array([3.1, 3.1]), np.array([5.5, 5.5])],
#     # "Wolf": [np.array([4, 4]), np.array([6, 6])]
#     # },
# }

fig, ax = plt.subplots(figsize=(10, 10))

def update(frame):
    ax.clear()
    small_world.step()

    species_present = set()

    for agent in small_world.agents:
        if agent.pos is None:
            continue

        colour = AGENT_COLOURS.get(agent.species, "green")
        size = 5 if agent.species in AGENT_COLOURS else 1

        ax.scatter(agent.pos[0], agent.pos[1], c=colour, s=size * 10, zorder=3)

        if agent.species in SENSING_RADIUS:
            circle = Circle(
                (agent.pos[0], agent.pos[1]),
                radius=SENSING_RADIUS[agent.species],
                edgecolor="green", facecolor="green",
                alpha=0.1, linewidth=1.5, linestyle="--", zorder=2
            )
            ax.add_patch(circle)

        if agent.species in HUNTING_RADIUS:
            hunt_circle = Circle(
                (agent.pos[0], agent.pos[1]),
                radius=HUNTING_RADIUS[agent.species],
                edgecolor="red", facecolor="red",
                alpha=0.7, linewidth=1.5, linestyle="--", zorder=2
            )
            ax.add_patch(hunt_circle)

        species_present.add(agent.species)

    legend_elements = []
    for species in AGENT_COLOURS:
        if species in species_present:
            legend_elements.append(
                Patch(
                    facecolor=AGENT_COLOURS[species], alpha=0.3,
                    edgecolor=AGENT_COLOURS[species],
                    label=f"{species} (r={SENSING_RADIUS.get(species, '?')})"
                )
            )
            if species in HUNTING_RADIUS:
                legend_elements.append(
                    Patch(
                        facecolor='red', alpha=0.7,
                        edgecolor='red',
                        label=f"{species} Hunting Radius (r={HUNTING_RADIUS.get(species, '?')})"
                    )
                )
    ax.legend(handles=legend_elements, loc="upper right", fontsize=14, framealpha=0.9)

    hours, mins = steps_to_time(small_world.steps)
    ax.set_xlim(0, small_world.space.x_max)
    ax.set_ylim(0, small_world.space.y_max)
    ax.set_aspect("equal")
    ax.set_title(f"Step: {hours}h {mins}m", fontsize=14)
    ax.set_xlabel("X (km)", fontsize=14)
    ax.set_ylabel("Y (km)", fontsize=14)
    ax.set_facecolor("#f5f5f0")
    ax.tick_params(axis='both', which='major', labelsize=12)
    fig.tight_layout()

    if frame % 50 == 0:
        print(f"Frame {frame}/200...")

small_world = SpeciesModel(**model_params)

print("Starting animation rendering...")
anim = FuncAnimation(fig, update, frames=200, interval=100, repeat=False)

# Save as GIF — no ffmpeg needed
print("Saving GIF...")
anim.save("predator_prey.gif", writer="pillow", fps=10)
plt.close(fig)
print(" Done! Open predator_prey.gif")

In [ ]:
from IPython.display import Image, display
display(Image(filename="predator_prey.gif"))

In [ ]:
# import matplotlib
# matplotlib.use('Agg')

# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from matplotlib.patches import Circle, Patch

# # Parameters
# model_params = {
#     "max_steps": 1500,
#     "init_predators": 7,
#     "init_deer": 30,
#     "height": 10,
#     "width": 10,
#     "step_size": 1/60,
#     "yearly_sunlight_hours": 365*24,
#     "seed": None,
#     "predator": 'Wolf',
#     "energy_decrease": 0.002,
#     "pack_limit": 12,
#     "use_base": False,
#     "use_pack_dynamics": True,
#     "use_random_movement": False,
#     "use_veg": False,
#     "given_positions": False,
#     "use_boundary_conditions": True,
#     "given_positions": {
#         "Deer": [np.array([5, 5]), np.array([10, 10]), np.array([3, 3]), np.array([9.5, 9.5])],
#         "Wolf": [np.array([4, 4]), np.array([11, 11])]
#     },
# }


# fig, ax = plt.subplots(figsize=(10, 10))

# def update(frame):
#     ax.clear()
#     small_world.step()

#     species_present = set()

#     for agent in small_world.agents:
#         if agent.pos is None:
#             continue

#         colour = AGENT_COLOURS.get(agent.species, "green")
#         size = 5 if agent.species in AGENT_COLOURS else 1

#         ax.scatter(agent.pos[0], agent.pos[1], c=colour, s=size * 10, zorder=3)

#         if agent.species in SENSING_RADIUS:
#             circle = Circle(
#                 (agent.pos[0], agent.pos[1]),
#                 radius=SENSING_RADIUS[agent.species],
#                 edgecolor="green", facecolor="green",
#                 alpha=0.1, linewidth=1.5, linestyle="--", zorder=2
#             )
#             ax.add_patch(circle)

#         if agent.species in HUNTING_RADIUS:
#             hunt_circle = Circle(
#                 (agent.pos[0], agent.pos[1]),
#                 radius=HUNTING_RADIUS[agent.species],
#                 edgecolor="red", facecolor="red",
#                 alpha=0.7, linewidth=1.5, linestyle="--", zorder=2
#             )
#             ax.add_patch(hunt_circle)

#         species_present.add(agent.species)

#     legend_elements = []
#     for species in AGENT_COLOURS:
#         if species in species_present:
#             legend_elements.append(
#                 Patch(
#                     facecolor=AGENT_COLOURS[species], alpha=0.3,
#                     edgecolor=AGENT_COLOURS[species],
#                     label=f"{species} (r={SENSING_RADIUS.get(species, '?')})"
#                 )
#             )
#     ax.legend(handles=legend_elements, loc="upper right", fontsize=10, framealpha=0.9)

#     hours, mins = steps_to_time(small_world.steps)
#     ax.set_xlim(0, small_world.space.x_max)
#     ax.set_ylim(0, small_world.space.y_max)
#     ax.set_aspect("equal")
#     ax.set_title(f"Step: {hours}h {mins}m", fontsize=14)
#     ax.set_xlabel("X (km)")
#     ax.set_ylabel("Y (km)")
#     ax.set_facecolor("#f5f5f0")
#     fig.tight_layout()

#     if frame % 50 == 0:
#         print(f"Frame {frame}/200...")

# small_world = SpeciesModel(**model_params)

# print("Starting animation rendering...")
# anim = FuncAnimation(fig, update, frames=200, interval=100, repeat=False)

# # Save as GIF — no ffmpeg needed
# print("Saving GIF...")
# anim.save("predator_prey_hunt.gif", writer="pillow", fps=10)
# plt.close(fig)
# print(" Done! Open predator_prey_hunt.gif")

### Deer Movement

In [ ]:
# Parameters to view deer fleeing sensing radius
model_params = {
    "max_steps": 1500,
    "init_predators": 2,
    "init_deer": 4,
    "height": 30,
    "width": 30,
    "step_size": 0.25,
    "yearly_sunlight_hours": 8760,
    "seed": None,
    "predator": "Wolf",
    "energy_decrease": 0.002,
    "pack_limit": 12,
    "veg_patch_spacing": 4,
    "sapling_density": 20,
    "tree_density": 8,
    "sapling_regrowth_prob": 1/10,
    "sapling_maturation_prob": 1/20000,
    "use_base": False,
    "use_pack_dynamics": True,
    "use_random_movement": False,
    "use_veg": False,
    "given_positions": {
        "Deer": [np.array([5, 5]), np.array([10, 10]), np.array([3, 3]), np.array([9.5, 9.5])],
        "Wolf": [np.array([4, 4]), np.array([11, 11])]
    },
    "use_boundary_conditions": True,
}

# Create the model
small_world = SpeciesModel(**model_params)

SolaraViz(
    small_world,
    components=[
        # make_space_component(agent_portrayal=agent_draw, backend="matplotlib"),
        SpaceWithSensing
    ],
    model_params=model_params,
    name="Predator-Prey Simulation",
)

### Static checks (IGNORE)

In [ ]:
def plot_headings(positions, headings, sensing_radius, animal, old_positions=np.array([]), old_headings=np.array([])):
    # Plot 
    fig, ax = plt.subplots(1,1)
    # Plot deer
    ax.scatter(deer_positions[:,0], deer_positions[:,1], c='blue', label='deer')
    # Plot wolves
    ax.scatter(positions[:,0], positions[:,1], c='red', label='wolf')

    # Plot headings
    arrow_scale = 1000    
    ax.quiver(
        positions[:,0], 
        positions[:,1], 
        headings[:,0] * arrow_scale, 
        headings[:,1] * arrow_scale, 
        width=0.003,                # Arrow shaft width
        headwidth=4,                # Arrow head width
        headlength=5,
        label='heading vector')
    
    # plot sensing radius
    for i in range(len(positions)):
        circ = plt.Circle((positions[i][0], positions[i][1]), radius=sensing_radius, edgecolor='b', facecolor='None')
        ax.add_patch(circ) 

    if len(old_headings) > 0 and len(old_positions) > 0:
        # Plot positions
        ax.scatter(old_positions[:,0], old_positions[:,1], c='red', label='previous position', alpha=0.4)
        
        ax.quiver(
            old_positions[:,0], 
            old_positions[:,1], 
            old_headings[:,0] * arrow_scale, 
            old_headings[:,1] * arrow_scale, 
            width=0.003,                # Arrow shaft width
            headwidth=4,                # Arrow head width
            headlength=5,
            alpha=0.4, 
            color='black',
            label='previous headings'
            )

    

    ax.set_title(f'Demonstrating {animal} movement')
    ax.legend(loc='upper right')
    plt.show()

In [ ]:
num_wolves = 5
num_deer = 5

# Create smaller world 
small_world = SpeciesModel(
    init_predators=num_wolves,
    init_deer = num_deer,
    height=30000,     
    width=30000,
    seed=40,
    init_num_of_packs = 3,
    predator = 'Wolf',  # Helper attribute to avoid imports when accessing agent type
    energy_decrease = 0.05,  # Energy decrease parameter 
    energy_min = 0,  # Point at which the animal will die of exhaustion
    veg_cell_size = 1,  # Introducing vegetation
    # Options to control complexity of the model
    use_pack_dynamics = True,  
    use_random_movement = False,
    use_veg = False
)

wolf_positions = []
deer_positions = []
wolves = []
deer = []
for agent in small_world.space.agents:
    x,y = agent.pos
    # check if wolf
    if agent.species == 'Wolf':
        wolf_positions.append([x,y])   
        wolves.append(agent)     
    elif agent.species == 'Deer':
        deer_positions.append([x,y])
        deer.append(agent)     

deer_positions = np.array(deer_positions)

wolf_headings = np.zeros((num_wolves,2))
for i,w in enumerate(wolves):
    wolf_headings[i] = w.heading


# Perform some steps
n_steps = 5
headings = np.zeros((n_steps,len(wolf_headings), 2))
positions = np.zeros((n_steps, len(wolf_positions),2))

for step in range(n_steps):
    # Move the wolves each step, storing their positions and headings
    for i,w in enumerate(wolves):
        w.move()
        headings[step][i] = w.heading
        x,y = w.pos
        positions[step][i] = [x,y]



plot_headings(positions[2], headings[2], wolves[0].sensing_radius, 'Wolf', positions[1], headings[1])
plot_headings(positions[4], headings[4], wolves[0].sensing_radius, 'Wolf', positions[3], headings[3])

